# Ayudantía 09: Serialización y networking II 🥒️ 🛜️

## Ayudantes 👾
Y sus recomendaciones semanales 🎵

- S1: Josefa Buch
  - [por definir](https://www.youtube.com/X)
- S2: Bastián Pérez
  - [por definir](https://www.youtube.com/X)
- S3: Carlos Martel
  - [por definir](https://www.youtube.com/X)
- S4: André Angulo
  - [por definir](https://www.youtube.com/X)
- S5: Juan Pablo Olivares
  - [por definir](https://www.youtube.com/X)
## Contenidos 📖

- Serialización y deserialización con pickle.
- Bytes, bytearray, encodings y formas de interpretar los bytes (little/big endian).
- Sockets TCP y UDP en python.
- Flujo completo desde serialización con pickle hasta el envío parcial por chunks de bytes.

## Resumen de Serialización y Networking 🥒🛜

### 🔑 Conceptos clave

| **Concepto** | **Qué es** |
|:--------------|:-------------|
| Serialización (`pickle`) | Convierte casi cualquier objeto de Python en *bytes* (`dumps`/`dump`) y los reconstruye después (`loads`/`load`); personalizable con `__getstate__`/`__setstate__` |
| `bytes` | Secuencia **inmutable** de *bytes*; se obtiene codificando un `str` (`encode`) o se interpreta como texto con `decode` |
| `bytearray` | Igual que `bytes`, pero **mutable**: se puede modificar, o usar `append`/`extend` como en una lista |
| *Encoding* | Forma de asociar *bytes* con caracteres (ASCII, UTF-8, *Latin-1*, ...); UTF-8 es compatible hacia atrás con ASCII |
| *Byteorder* (`big`/`little`) | Orden en que se guardan los *bytes* de un número: en `big` el *byte* más significativo va primero |
| *Chunk* | Fragmento de *bytes* que se lee/envía de a poco (ej. de 4096 *bytes*), en vez de todo de una vez o *byte* por *byte* |
| JSON / Base64 | JSON serializa estructuras simples (`dict`, `list`, `str`, `int`, ...) a texto; Base64 convierte *bytes* arbitrarios (ej. un archivo) en texto para que puedan ir dentro de un JSON |
| *Socket* | Objeto del sistema operativo para enviar y recibir datos por la red; se crea indicando familia de direcciones (`AF_INET`) y protocolo de transporte (`SOCK_STREAM` para TCP, `SOCK_DGRAM` para UDP) |

### ⚙️ Comandos y sintaxis más usados

| **Categoría** | **Comando / Sintaxis** | **Descripción breve** | **Ejemplo** |
|:--------------|:-----------------------|:-------------------------|:--------------|
| Crear *socket* | `socket.socket(family, type)` | Crea un *socket* TCP (`SOCK_STREAM`) o UDP (`SOCK_DGRAM`) | `socket.socket(socket.AF_INET, socket.SOCK_STREAM)` |
| Servidor: enlazar | `sock.bind((host, port))` | Asocia el *socket* a un *host* y puerto donde escuchará conexiones | `sock.bind(('localhost', 8998))` |
| Servidor: escuchar | `sock.listen(n)` | Empieza a aceptar hasta `n` conexiones pendientes | `sock.listen(3)` |
| Servidor: aceptar | `sock.accept()` | Bloquea hasta que llega un cliente; retorna `(socket_cliente, address)` | `cliente, addr = sock.accept()` |
| Cliente: conectar | `sock.connect((host, port))` | Establece la conexión con el servidor | `sock.connect(('localhost', 8998))` |
| Enviar *bytes* | `sock.sendall(bytes)` | Envía **todos** los *bytes* del mensaje (reintenta internamente hasta completar) | `sock.sendall(datos)` |
| Recibir *bytes* | `sock.recv(buffer)` | Lee **hasta** `buffer` *bytes* disponibles en el *socket* (puede entregar menos) | `sock.recv(4096)` |
| Cerrar | `sock.close()` | Libera el *socket*; siempre debe llamarse al terminar | `sock.close()` |
| Texto → *bytes* | `texto.encode('utf-8')` | Codifica un `str` a `bytes` según un *encoding* | `'hola'.encode('utf-8')` |
| *Bytes* → texto | `datos.decode('utf-8')` | Decodifica `bytes` a `str` según un *encoding* | `datos.decode('utf-8')` |
| Entero → *bytes* | `n.to_bytes(largo, byteorder='big')` | Convierte un `int` en una cantidad fija de *bytes* | `len(datos).to_bytes(4, 'big')` |
| *Bytes* → entero | `int.from_bytes(datos, byteorder='big')` | Convierte *bytes* en el `int` que representan | `int.from_bytes(sock.recv(4), 'big')` |
| Serializar (`pickle`) | `pickle.dumps(obj)` / `pickle.loads(bytes)` | Convierte un objeto a *bytes* y viceversa | `pickle.dumps(persona)` |
| Serializar (JSON) | `json.dumps(dict)` / `json.loads(str)` | Convierte un `dict`/`list` a `str` (y viceversa); luego hay que `encode`/`decode` para obtener *bytes* | `json.dumps({"comando": "carga"})` |
| Binario → texto | `base64.b64encode(bytes)` / `base64.b64decode(str)` | Representa *bytes* arbitrarios (ej. un archivo) como texto seguro para incluir en un JSON | `base64.b64encode(contenido).decode()` |
| Múltiples clientes | `threading.Thread(target=f, args=(...))` | Corre `f` en paralelo para no bloquear al resto del servidor mientras se atiende a un cliente | `threading.Thread(target=self.client_connection, args=(sock, addr)).start()` |

> 💡 **El protocolo de este ejercicio:** tanto `send_data` del servidor como `send_json` del cliente deben enviar primero 4 *bytes* con el largo del mensaje (`len(mensaje).to_bytes(4, 'big')`) y luego el mensaje en sí. Del otro lado, primero se lee con `recv(4)` y se traduce con `int.from_bytes(..., 'big')` para saber cuántos *bytes* hay que seguir leyendo.

> ⚠️ **`recv` no garantiza todo de una vez:** cada llamada a `recv(buffer)` puede entregar menos *bytes* de los pedidos. Por eso hay que acumular lo recibido (por ejemplo en un `bytearray`) hasta alcanzar el largo esperado, en vez de asumir que un solo `recv` basta para leer el mensaje completo.

## DCClassroomServer 🏫

Luego de desarrollar DCCPuchamon te has dado cuenta que la comunicación entre cliente y servidor no es tan simple como parece. Por lo que decides crear una nueva aplicación llamada DCClassroomServer, la cual permitirá a los estudiantes conectarse a un servidor centralizado en el que podrán subir sus tareas, descargar material de estudio, ver a la lista de estudiantes del curso y comunicarse con ellos.

Para lograrlo, debes tener en cuenta el protocolo de aplicación que usarás entre el cliente y el servidor.

El cliente debe enviar **solicitudes** al servidor en formato JSON, y con los siguientes atributos:

  - `username`: Nombre del usuario del estudiante que realiza la solicitud.
  
  - `comando`: Indica la operación que se desea realizar. Puede ser ``'carga'`` (subir un archivo), ``'descarga'`` (descargar un archivo), ``'listar_usuarios'`` (obtener la lista de estudiantes conectados) o ``'mensaje_directo'`` (enviar un mensaje a otro estudiante).

  - `file_name`: Nombre del archivo a subir o descargar. Incluir en el caso de las operaciones ``'listar_usuarios'`` y ``'mensaje_directo'``.

  - `file_content_b64`: Contenido del archivo codificado en base64 _(*)_. Solo aplicable para la operación ``'carga'``.
  
  - `destinatario`: Nombre del usuario al que se desea enviar un mensaje. Solo aplicable para la operación ``'mensaje_directo'``.
  
  - `mensaje`: Mensaje a enviar al estudiante. Solo incluir para la operación ``'mensaje_directo'``.

El servidor debe **responder** a las solicitudes del cliente de la siguiente forma:

- Si se lanza un error, debe un JSON con los siguientes atributos:
  - `tipo`: Indica el tipo de respuesta. En este caso, siempre será ``'error'``.
  - `descripcion`: Mensaje descriptivo del error ocurrido.

- Si todo sale bien, el servidor debe responder con un JSON con los siguientes atributos:
  
  - Si el comando fue ``'carga'``, el servidor debe responder con:
    - `tipo`: ``'confirmacion'``.
    - `descripcion`: Mensaje confirmando que el archivo fue recibido correctamente.
  
  - Si el comando fue ``'descarga'``, el servidor debe responder con:
    - `username`: Corresponde a ``'Servidor'``.
    - `tipo`: ``'archivo_descargado'``.
    - `file_name`: Nombre del archivo descargado.
    - `file_content_b64`: Contenido del archivo descargado codificado en base64 _(*)_.
  
  - Si el comando fue ``'listar_usuarios'``, el servidor debe responder con:
    - `username`: Corresponde a ``'Servidor'``.
    - `tipo`: ``'lista_usuarios'``.
    - `usuarios`: Lista de nombres de usuarios conectados.
  
  - Si el comando fue ``'mensaje_directo'``, el servidor debe enviar al destinatario un JSON con:
    - `username`: Corresponde a ``'Servidor'``.
    - `tipo`: ``'mensaje_entrante'``.
    - `remitente`: Nombre del usuario que envió el mensaje.
    - `mensaje`: Mensaje recibido.
    Y responder al remitente con:
    - `tipo`: ``'confirmacion'``.
    - `descripcion`: Mensaje confirmando que el mensaje fue enviado correctamente.

_(*) Se utiliza base64 para codificar los datos binarios a una representación de texto que pueda ser incluída en el JSON._

## Ejercicio 1: Servidor 🖥️

Deberás completar la clase `Server` (en el archivo `server.py`) para manejar la conexión con cada estudiante. Asegúrate de manejar correctamente los errores y cerrar las conexiones una vez que la operación haya finalizado. Especificamente deberas completar:
- **`client_connection(self, client_socket: socket.socket, addr: tuple)`**:
  
  - Este método debe escuchar la conexión del cliente, recibir el tamaño del mensaje, los bytes de la request, deserializar el JSON y manejar los diferentes comandos (`carga`, `descarga`, `listar_usuarios`, `mensaje_directo`) según corresponda. Finalmente, debe cerrar el socket del cliente y eliminar su información de la lista de clientes conectados. En cada comando se debe realizar lo siguiente:
  
    - En el caso de `carga`, se debe verificar que se hayan incluído los atributos `file_name` y `file_content_b64`. Si alguno falta, se debe enviar un mensaje de error al cliente. Si ambos están presentes, se debe decodificar el contenido del archivo desde base64 y guardarlo como archivo en el servidor.
  
    - En el caso de `descarga`, se debe verificar que el archivo solicitado exista en el servidor. Si no existe, se debe enviar un mensaje de error al cliente. Si existe, se debe leer el archivo, codificar su contenido en base64 y enviarlo al cliente.
  
    - Si el comando es `listar_usuarios`, se debe enviar al cliente la lista de usuarios conectados.
  
    - Si el comando es `mensaje_directo`, se hayan incluído los atributos `destinatario` y `mensaje`. Si alguno falta, se debe enviar un mensaje de error al cliente. Si ambos están presentes, se debe obtener el socket del destinatario y enviarle el mensaje directo. Luego, se debe enviar una confirmación al remitente.
  
- **`send_data(self, data: str, client_socket: socket.socket)`**:

  - Este método debe pasar la data a enviar a bytes, obtener su tamaño y enviar primero el tamaño (4 bytes) y luego la data como tal.

## Ejercicio 2: Cliente 💻

Ademas deberas completar por la parte del `Cliente` el cual buscara conectarse al servidor y enviar diversos tipos de mensajes especificos mediante la consola, especificamente:
- Enviar y recibir archivos (`subir` / `descargar`).
- Solicitar y ver la lista de todos los alumnos conectados (`lista`).
- Enviar mensajes directos a un alumno específico (`msg`).
- Identificarse ante el servidor al conectarse (esto lo hace `connect_to_server` al enviar el `"saludo"`, que a su vez usa `send_json`).

A continuación, se explica la lógica de las funciones que implementan esto:

- **`send_json(self, msg: dict)`**
    - Esta es la función central de envío. Se encarga de convertir el diccionario de Python (`msg`) en un string JSON, codificarlo a bytes (`UTF-8`), calcular su longitud, y enviar primero la longitud (4 bytes) y luego los datos (el JSON) al servidor. Siempre se asegura de que el `username` del cliente (`self.nombre_cliente`) esté en el mensaje antes de enviarlo.

- **`mandar_solicitud_archivo(self, tipo, nombre_archivo)`**
    - **Si `tipo == "carga"`:** Esta parte lee un archivo local (`nombre_archivo`) desde el disco en modo binario (`'rb'`). Luego, codifica esos bytes en Base64 (un formato de texto seguro para JSON) y lo envía al servidor dentro de un JSON con la operación `"carga"`, el nombre del archivo (`file_name`) y el contenido ya codificado en `"file_content_b64"`.
    - **Si `tipo == "descarga"`:** Esta parte es más simple. Solo envía un JSON al servidor con la operación `"descarga"` y el `file_name` que se quiere obtener. El servidor buscará el archivo y, si lo encuentra, responderá con un mensaje `archivo_descargado`, que será procesado por `decode_msg_from_server`.

- **`mandar_solicitud_alumnos(self)`**
    - Esta función construye y envía un JSON muy simple al servidor. El JSON solo contiene la operación `"listar_usuarios"`. El servidor, al recibir esto, responderá con un mensaje tipo `"lista_usuarios"` que incluye a todos los clientes conectados, y `decode_msg_from_server` lo imprimirá en la consola.

- **`mandar_mensaje_alumno(self, destinado, msg)`**
    - Esta función empaqueta un JSON para el servidor con la operación `"mensaje_directo"`, el nombre del `"destinatario"` (a quién va el mensaje) y el `"mensaje"` en sí. El servidor se encarga de buscar el socket de ese destinatario y reenviarle el mensaje.

_Nota: Los mensajes deben enviarse con endianess 'big'._

## Ejercicio 3: Modelando un sistema en red 🗺️

**DCCFotos** es un sistema donde los usuarios suben fotos a un servidor y otros las descargan. Sus entidades son:
- **Cliente 1**: app con interfaz gráfica que sube fotos.
- **Cliente 2**: app de consola que descarga fotos.
- **Servidor**: programa en Python que atiende a los clientes con sockets TCP.
- **`fotos.pkl`**: archivo en disco donde el servidor guarda las fotos, en un diccionario serializado con `pickle`.

El sistema funciona así:
1. El Cliente 1 sube una foto: envía su nombre y tamaño, y luego su contenido. El servidor la guarda en `fotos.pkl` y responde con una confirmación.
2. El Cliente 2 pide una foto por su nombre. El servidor la lee de `fotos.pkl` y se la envía. Si la foto no existe, responde con un error.

Dibuja un diagrama de este sistema, con las entidades como cajas y los mensajes como flechas numeradas en orden. Cada flecha debe indicar **qué información viaja**, su **formato** (JSON, `pickle`, bytes, largo previo, chunks, etc.) y el **canal** (socket TCP, disco). Distingue las respuestas de las peticiones, por ejemplo con flechas punteadas.

Dibújalo con la herramienta que prefieras y pégalo aquí:


**Solución de referencia**

![Diagrama de referencia](diagrama_solucion.png)

| # | De → a | Qué viaja | Formato | Canal |
|:--|:--|:--|:--|:--|
| 1 | Cliente 1 → Servidor | Nombre y tamaño de la foto | JSON en UTF-8, precedido por su largo en 4 bytes (big endian) | Socket TCP |
| 2 | Cliente 1 → Servidor | Contenido de la foto | Bytes, enviados en chunks de 4096 | Socket TCP |
| 3 | Servidor → `fotos.pkl` | La foto, dentro del diccionario | Diccionario serializado con `pickle.dump` | Escritura en disco |
| 4 | Servidor → Cliente 1 | Confirmación | JSON en UTF-8, precedido por su largo en 4 bytes | Socket TCP |
| 5 | Cliente 2 → Servidor | Nombre de la foto pedida | JSON en UTF-8, precedido por su largo en 4 bytes | Socket TCP |
| 6 | `fotos.pkl` → Servidor | Diccionario con las fotos | Bytes deserializados con `pickle.load` | Lectura en disco |
| 7 | Servidor → Cliente 2 | Tamaño y contenido de la foto | Tamaño en 4 bytes, y luego los bytes en chunks de 4096 | Socket TCP |
| 7b | Servidor → Cliente 2 | Error: la foto no existe | JSON en UTF-8, precedido por su largo en 4 bytes | Socket TCP |


## Ejercicio 4: Preguntas de alternativa

![P14(2025-2)](preguntas_alternativas/p1.png)

### Respuesta correcta
`P1`: B

![P16(2025-2)](preguntas_alternativas/p2.png)

### Respuesta correcta
`P2`: D

![P30(2025-2)](preguntas_alternativas/p3.png)

### Respuesta correcta
`P3`: E

![P10(2025-1)](preguntas_alternativas/p4.png)

### Respuesta correcta
`P4`: E

![P16(2025-1)](preguntas_alternativas/p5.png)

### Respuesta correcta
`P5`: A




## Bonus: DCCGift 🎁

El equipo docente te preparó un regalo para terminar la ayudantía 😁

Está guardado en el archivo `dcc_gift.pkl`. Cárgalo con `pickle` y descubre qué es:

In [ ]:
import pickle

with open("dcc_gift.pkl", "rb") as f:
    regalo = pickle.load(f)